# Montagem de novo com SPAdes

Objetivo: transformar reads paired-end processados em contigs e calcular
estatísticas básicas da montagem.

> O artigo de *Hypochilus* utilizou Trinity + Velvet. Nesta disciplina usamos
> SPAdes como estratégia didática para observar claramente a etapa de montagem.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
QC = ROOT / "04_qc_trimming"
BASE = ROOT / "05_spades"
OUT = BASE / "SRR15736591"

R1 = QC / "SRR15736591_R1_paired.fastq.gz"
R2 = QC / "SRR15736591_R2_paired.fastq.gz"

assert R1.exists() and R2.exists(), "Execute primeiro o notebook de QC/trimming."
BASE.mkdir(parents=True, exist_ok=True)
print(R1, R2, sep="\n")

## 1. Instalar SPAdes

In [ ]:
!apt-get -qq update
!apt-get -qq install -y spades
!spades.py --version

## 2. Executar a montagem

In [ ]:
!rm -rf "$OUT"
!spades.py   --only-assembler   --careful   -1 "$R1"   -2 "$R2"   -o "$OUT"   -t 2   -m 12

## 3. Confirmar os arquivos

In [ ]:
!ls -lh "$OUT/contigs.fasta" "$OUT/scaffolds.fasta"

## 4. Calcular estatísticas da montagem

In [ ]:
from pathlib import Path

contigs = OUT / "contigs.fasta"

def fasta_lengths(path):
    lengths = []
    seq = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if seq:
                    lengths.append(len("".join(seq)))
                seq = []
            else:
                seq.append(line)
        if seq:
            lengths.append(len("".join(seq)))
    return lengths

lens = sorted(fasta_lengths(contigs), reverse=True)
total = sum(lens)

def n50(lengths):
    half = sum(lengths)/2
    acc = 0
    for L in sorted(lengths, reverse=True):
        acc += L
        if acc >= half:
            return L

print("Número de contigs:", len(lens))
print("Comprimento total:", total)
print("Maior contig:", max(lens) if lens else 0)
print("N50:", n50(lens) if lens else 0)
print("Contigs >= 200 bp:", sum(L >= 200 for L in lens))

## 5. Filtrar contigs com pelo menos 200 bp

In [ ]:
filtered = OUT / "contigs_min200.fasta"

with open(contigs) as inp, open(filtered, "w") as out:
    header = None
    seq = []
    def flush():
        if header is not None:
            s = "".join(seq)
            if len(s) >= 200:
                out.write(header + "\n")
                for i in range(0, len(s), 80):
                    out.write(s[i:i+80] + "\n")
    for line in inp:
        line = line.rstrip("\n")
        if line.startswith(">"):
            flush()
            header = line
            seq = []
        else:
            seq.append(line.strip())
    flush()

print(filtered)

## 6. Preparar a montagem para PHYLUCE

O notebook seguinte utilizará `contigs_min200.fasta`.